# Phase 3.4 : Tableau de comparaison cross-modèles

Chats vs chiens (Microsoft Cats vs Dogs Dataset). Suite de `phase3_3_finetuning.ipynb`.

**Important — ce notebook n'est PAS autonome comme les précédents** : il agrège les résultats des phases 1.4, 2.2 et 3.3. Il a besoin de `history_scratch.json`, `history_augmented.json`, `history_tl_head.json`, `history_tl_finetune.json`, `model_scratch.keras`, `model_augmented.keras` et `model_tl.keras`.

**Deux façons de les obtenir** :
1. **Même runtime Colab** (recommandé) : exécute `phase1_4_training.ipynb`, `phase2_2_dropout.ipynb`, `phase3_2_head_training.ipynb` et `phase3_3_finetuning.ipynb` dans ce même environnement, sans redémarrer.
2. **Runtime différent** : télécharge ces 7 fichiers depuis les sessions précédentes et uploade-les ici.

In [ ]:
import os

required = [
    "history_scratch.json", "history_augmented.json",
    "history_tl_head.json", "history_tl_finetune.json",
    "model_scratch.keras", "model_augmented.keras", "model_tl.keras",
]
missing = [f for f in required if not os.path.exists(f)]

if missing:
    print("Fichiers manquants :", missing)
    print("Execute les phases 1.4, 2.2, 3.2 et 3.3 dans ce runtime avant de continuer,")
    print("ou uploade-les manuellement (cellule suivante).")
else:
    print("Tous les fichiers necessaires sont presents.")

In [ ]:
# A executer seulement si la cellule precedente signale des fichiers manquants.
from google.colab import files
files.upload()

In [ ]:
import json
import os
import tensorflow as tf

with open("history_scratch.json") as f:
    history_scratch_dict = json.load(f)
with open("history_augmented.json") as f:
    history_augmented_dict = json.load(f)
with open("history_tl_head.json") as f:
    history_tl_head_dict = json.load(f)
with open("history_tl_finetune.json") as f:
    history_tl_finetune_dict = json.load(f)

model_scratch = tf.keras.models.load_model("model_scratch.keras")
model_augmented = tf.keras.models.load_model("model_augmented.keras")
model_tl = tf.keras.models.load_model("model_tl.keras")


def model_size_mb(path):
    return os.path.getsize(path) / (1024 * 1024)


size_scratch = model_size_mb("model_scratch.keras")
size_augmented = model_size_mb("model_augmented.keras")
size_tl = model_size_mb("model_tl.keras")

## Phase 3.4 : Tableau de comparaison cross-modèles

**Objectif** : extraire les métriques clés des trois modèles et produire le tableau de comparaison complet qui documente la progression de la journée.

In [ ]:
val_acc_scratch = max(history_scratch_dict["val_accuracy"])
val_acc_augmented = max(history_augmented_dict["val_accuracy"])
val_acc_tl = max(history_tl_head_dict["val_accuracy"] + history_tl_finetune_dict["val_accuracy"])

params_scratch = model_scratch.count_params()
params_augmented = model_augmented.count_params()
params_tl = model_tl.count_params()

print("=" * 70)
print("TABLEAU DE COMPARAISON DES TROIS MODELES")
print("=" * 70)
print(f"{'Iteration':<25} {'val_acc':>8} {'Params':>12} {'Taille':>10}")
print("-" * 70)
print(f"{'CNN scratch':<25} {val_acc_scratch:>7.1%} {params_scratch:>12,} {size_scratch:>9.1f}Mo")
print(f"{'CNN augmente + Dropout':<25} {val_acc_augmented:>7.1%} {params_augmented:>12,} {size_augmented:>9.1f}Mo")
print(f"{'MobileNetV2 fine-tuning':<25} {val_acc_tl:>7.1%} {params_tl:>12,} {size_tl:>9.1f}Mo")
print("=" * 70)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
plt.plot(history_scratch_dict["val_accuracy"], label="CNN scratch", color="red", linestyle="--")
plt.plot(history_augmented_dict["val_accuracy"], label="Augmente + Dropout", color="orange")

tl_acc = history_tl_head_dict["val_accuracy"] + history_tl_finetune_dict["val_accuracy"]
plt.plot(range(len(tl_acc)), tl_acc, label="MobileNetV2 fine-tuning", color="green")

plt.xlabel("Epoch")
plt.ylabel("Val Accuracy")
plt.legend()
plt.title("Comparaison des trois iterations")
plt.tight_layout()
plt.savefig("comparison_all.png", dpi=100)
plt.show()

### À toi de rédiger (dans cette cellule)

À partir du tableau et du graphe ci-dessus :
- Qu'est-ce que le tableau révèle sur le rapport performance/coût de chaque approche ?
- Pourquoi MobileNetV2 peut gagner sur (presque) tous les fronts malgré un nombre de paramètres *entraînables* bien plus faible que le CNN from scratch ?

*(remplace ce texte par ton interprétation)*

### Checkpoint transfer learning

- **Happy path** : tableau à 3 lignes complet, graphe de superposition sauvegardé, paragraphe d'interprétation rédigé. Progression visible entre les trois modèles.
- **Edge case** : une colonne "temps par epoch" plutôt que "temps total" montrerait que MobileNetV2 va plus vite par epoch que le CNN scratch malgré 2.2M de paramètres gelés — la backprop ne traverse pas la base gelée.
- **Adversarial** : comparer `val_acc` seule, sans les paramètres ni la taille, ferait croire que TP3 est "magique" — le tableau doit toujours montrer le coût (params, taille) à côté du gain (accuracy), sinon il ment par omission.

**Prochaine étape** : Phase 3.5 — export TFLite et exploration ouverte.